## 1. Setup & Load Clean Data

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.components.data_ingestion import DataIngestion
from src.components.data_preprocessing import LogPreprocessor

In [2]:
ingestion = DataIngestion()
ingestion.start_spark_session()

df = ingestion.load_data("data/raw/HDFS.log")
# # LIMIT FIRST (before any heavy op)
# df = df.limit(500000)   # 5 lakh rows only

preprocessor = LogPreprocessor()
parsed_df = preprocessor.parse_logs(df)

from pyspark.sql.functions import col, concat_ws, to_timestamp

clean_df = parsed_df.filter(
    (parsed_df.date != "") &
    (parsed_df.time != "") &
    (parsed_df.log_level != "")
).withColumn("process_id", col("process_id").cast("int"))

clean_df = clean_df.withColumn(
    "datetime",
    to_timestamp(concat_ws(" ", clean_df.date, clean_df.time), "yyMMdd HHmmss")
)

clean_df.cache()
print("Clean Data Count:", clean_df.count())

Resolved path: c:\Users\Priyanshu\Desktop\Main\ML\Distributed-Log-Analysis\data\raw\HDFS.log
Clean Data Count: 11175629


## 2. Log Level Distribution

In [3]:
log_level_dist = clean_df.groupBy("log_level").count().orderBy("count", ascending=False)
log_level_dist.show()

+---------+--------+
|log_level|   count|
+---------+--------+
|     INFO|10812836|
|     WARN|  362793|
+---------+--------+



## 3. Top Components by Log Volume

In [4]:
# Use sampling to avoid JVM crash (important for local machine)
sample_df = clean_df.sample(fraction=0.1, seed=42)

# Compute top components
component_activity = sample_df.groupBy("component") \
    .count() \
    .orderBy("count", ascending=False)

# Show top 10
component_activity.show(10, truncate=False)

+------------------------------------------------------+------+
|component                                             |count |
+------------------------------------------------------+------+
|dfs.FSNamesystem                                      |369993|
|dfs.DataNode$PacketResponder                          |341529|
|dfs.DataNode$DataXceiver                              |252367|
|dfs.FSDataset                                         |141313|
|dfs.DataBlockScanner                                  |11926 |
|dfs.DataNode                                          |681   |
|dfs.DataNode$DataTransfer                             |673   |
|dfs.DataNode$BlockReceiver                            |183   |
|dfs.PendingReplicationBlocks$PendingReplicationMonitor|3     |
+------------------------------------------------------+------+



In [5]:
component_activity = clean_df.groupBy("component").count().orderBy("count", ascending=False)
component_activity.show(10, truncate=False)

+------------------------------------------------------+-------+
|component                                             |count  |
+------------------------------------------------------+-------+
|dfs.FSNamesystem                                      |3700245|
|dfs.DataNode$PacketResponder                          |3413350|
|dfs.DataNode$DataXceiver                              |2518678|
|dfs.FSDataset                                         |1407597|
|dfs.DataBlockScanner                                  |120046 |
|dfs.DataNode                                          |7002   |
|dfs.DataNode$DataTransfer                             |6946   |
|dfs.DataNode$BlockReceiver                            |1718   |
|dfs.PendingReplicationBlocks$PendingReplicationMonitor|47     |
+------------------------------------------------------+-------+



## 4. Top WARNING Generating Components (Failure Signal)

In [6]:
warn_df = clean_df.filter(clean_df.log_level == "WARN")

top_warn_components = warn_df.groupBy("component").count().orderBy("count", ascending=False)

top_warn_components.show(10, truncate=False)

+------------------------------------------------------+------+
|component                                             |count |
+------------------------------------------------------+------+
|dfs.DataNode$DataXceiver                              |356207|
|dfs.FSDataset                                         |5545  |
|dfs.FSNamesystem                                      |975   |
|dfs.PendingReplicationBlocks$PendingReplicationMonitor|47    |
|dfs.DataBlockScanner                                  |10    |
|dfs.DataNode$DataTransfer                             |9     |
+------------------------------------------------------+------+



## 5. Time-Based Log Analysis (Hourly)

In [7]:
from pyspark.sql.functions import hour

time_analysis = clean_df.withColumn("hour", hour("datetime")) \
    .groupBy("hour", "log_level") \
    .count() \
    .orderBy("hour")

time_analysis.show(24)

+----+---------+-------+
|hour|log_level|  count|
+----+---------+-------+
|   0|     WARN|   8039|
|   0|     INFO| 194221|
|   1|     WARN|   7961|
|   1|     INFO| 453056|
|   2|     INFO| 466716|
|   2|     WARN|  11522|
|   3|     WARN|   8053|
|   3|     INFO| 478482|
|   4|     WARN|   8482|
|   4|     INFO| 862427|
|   5|     INFO| 487799|
|   5|     WARN|   7910|
|   6|     WARN|   6612|
|   6|     INFO| 667349|
|   7|     WARN|  28712|
|   7|     INFO| 724619|
|   8|     INFO| 645035|
|   8|     WARN|  32578|
|   9|     INFO| 605789|
|   9|     WARN|  18169|
|  10|     INFO|1278047|
|  10|     WARN|   1425|
|  11|     WARN|     97|
|  11|     INFO| 690570|
+----+---------+-------+
only showing top 24 rows



## 6. Failure Pattern Detection (KEY PART)
Idea: Spike in WARN logs = anomaly

In [8]:
warn_time = warn_df.withColumn("hour", hour("datetime")) \
    .groupBy("hour") \
    .count() \
    .orderBy("count", ascending=False)

warn_time.show()

+----+-----+
|hour|count|
+----+-----+
|  13|65977|
|  22|57468|
|   8|32578|
|  21|31163|
|   7|28712|
|  12|23713|
|   9|18169|
|   2|11522|
|  23|10383|
|   4| 8482|
|   3| 8053|
|   0| 8039|
|  17| 7970|
|   1| 7961|
|   5| 7910|
|  18| 7846|
|  19| 7821|
|   6| 6612|
|  16| 5595|
|  20| 3223|
+----+-----+
only showing top 20 rows



## 7. Frequent Error Messages (Pattern Mining)

In [9]:
top_messages = warn_df.groupBy("message") \
    .count() \
    .orderBy("count", ascending=False)

top_messages.show(10, truncate=False)

+--------------------------------------------------------------------------------------------+-----+
|message                                                                                     |count|
+--------------------------------------------------------------------------------------------+-----+
|10.250.14.224:50010:Got exception while serving blk_-1347702416509823842 to /10.250.14.224: |15   |
|10.251.215.192:50010:Got exception while serving blk_8512415457710305414 to /10.251.215.192:|13   |
|10.251.30.179:50010:Got exception while serving blk_2356980171746022239 to /10.251.30.179:  |13   |
|10.251.194.213:50010:Got exception while serving blk_3584987098972940224 to /10.251.194.213:|10   |
|10.251.126.255:50010:Got exception while serving blk_548447233881510444 to /10.251.126.255: |10   |
|10.250.14.224:50010:Got exception while serving blk_-4701591807520324601 to /10.250.14.224: |9    |
|10.251.198.33:50010:Got exception while serving blk_3568358184216233747 to /10.251.198.33:

## 8. Node/Process Failure Analysis

In [10]:
process_issues = warn_df.groupBy("process_id") \
    .count() \
    .orderBy("count", ascending=False)

process_issues.show(10)

+----------+-----+
|process_id|count|
+----------+-----+
|        19| 4769|
|        18|  776|
|      3403|  137|
|      3442|  136|
|      3421|  136|
|      3411|  135|
|     11729|  133|
|      3424|  131|
|      3439|  131|
|      3567|  131|
+----------+-----+
only showing top 10 rows



## 9. Failure Threshold Detection (Simple Prediction Logic)
Define failure condition:

👉 If WARN count > threshold → failure risk

In [11]:
from pyspark.sql.functions import when

threshold = 50  # you can tune this

warn_hourly = warn_df.withColumn("hour", hour("datetime")) \
    .groupBy("hour") \
    .count()

failure_prediction = warn_hourly.withColumn(
    "failure_risk",
    when(col("count") > threshold, "HIGH").otherwise("NORMAL")
)

failure_prediction.show()

+----+-----+------------+
|hour|count|failure_risk|
+----+-----+------------+
|  20| 3223|        HIGH|
|  23|10383|        HIGH|
|  22|57468|        HIGH|
|   0| 8039|        HIGH|
|  21|31163|        HIGH|
|   3| 8053|        HIGH|
|   4| 8482|        HIGH|
|   8|32578|        HIGH|
|   7|28712|        HIGH|
|   5| 7910|        HIGH|
|   2|11522|        HIGH|
|   9|18169|        HIGH|
|   1| 7961|        HIGH|
|   6| 6612|        HIGH|
|  11|   97|        HIGH|
|  10| 1425|        HIGH|
|  14| 2027|        HIGH|
|  12|23713|        HIGH|
|  13|65977|        HIGH|
|  19| 7821|        HIGH|
+----+-----+------------+
only showing top 20 rows



## 10. Component-Level Failure Risk

In [12]:
component_risk = warn_df.groupBy("component") \
    .count() \
    .withColumn(
        "risk_level",
        when(col("count") > 100, "HIGH")
        .when(col("count") > 50, "MEDIUM")
        .otherwise("LOW")
    ) \
    .orderBy("count", ascending=False)

component_risk.show(10, truncate=False)

+------------------------------------------------------+------+----------+
|component                                             |count |risk_level|
+------------------------------------------------------+------+----------+
|dfs.DataNode$DataXceiver                              |356207|HIGH      |
|dfs.FSDataset                                         |5545  |HIGH      |
|dfs.FSNamesystem                                      |975   |HIGH      |
|dfs.PendingReplicationBlocks$PendingReplicationMonitor|47    |LOW       |
|dfs.DataBlockScanner                                  |10    |LOW       |
|dfs.DataNode$DataTransfer                             |9     |LOW       |
+------------------------------------------------------+------+----------+

